<a href="https://colab.research.google.com/github/chidiview-ui/shiny-telegram/blob/main/RealTime_Currency_Conversion_Tool_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.3 MB/s 

In [ ]:
from google.colab import userdata
import os

OPEN_API_KEY = userdata.get("OpenAi")
os.environ["OPENAI_API_KEY"] = OPEN_API_KEY

In [ ]:
SERPER_API_KEY = userdata.get("Serper_key")
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

In [ ]:
EXCHANGE_RATE_API_KEY = userdata.get("ExchangeRate")
os.environ["EXCHANGE_RATE_API_KEY"] = EXCHANGE_RATE_API_KEY

In [ ]:

#os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
#o#s.environ["EXCHANGE_RATE_API_KEY"] = EXCHANGE_RATE_API_KEY

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
import requests
from typing import Type
from pydantic import BaseModel,Field
from crewai.tools import BaseTool


In [ ]:
# Define Input fields the tools expect using Pydantic
class CurrencyConverterToolSchema(BaseModel):
    """Input Schema for CurrencyConverterTool """
    amount: float = Field(..., description="The amount to convert.")
    from_currency: str = Field(..., description='The source currency code (e.g., "USD")')
    to_currency: str = Field(..., description='The target currency code (e.g., "EUR")')

In [ ]:
# Now we define the Currency Converting Tool by Inheriting from Basetool:
class CurrencyConverterTool(BaseTool):
    name: str = "Currency Converter Tool"
    description: str = "Converts one currency to another."
    args_schema: Type[BaseModel] = CurrencyConverterToolSchema
    api_key: str = os.getenv("EXCHANGE_RATE_API_KEY")

    def _run(self, amount: float, from_currency: str, to_currency: str) -> str:
        url = f"https://v6.exchangerate-api.com/v6/{self.api_key}/pair/{from_currency}/{to_currency}"
        response = requests.get(url)

        if response.status_code != 200:
            return "Failed to fetch exchange rates."

        data = response.json()
        if "conversion_rate" not in data:
             return f"Could not find conversion rate for {from_currency} to {to_currency}."


        rate = data["conversion_rate"]
        converted_amount = amount * rate
        return f"{amount} {from_currency} is equal to {converted_amount: .2f} {to_currency}"

In [ ]:
currency_tool = CurrencyConverterTool()
print(currency_tool.run(amount=100, from_currency="USD", to_currency="EUR"))

Using Tool: Currency Converter Tool
100 USD is equal to  85.42 EUR


In [ ]:
import os
from crewai import Agent
from crewai.tools import BaseTool
from pydantic.v1 import BaseModel, Field
from typing import Type

In [ ]:
from crewai import Agent
currency_analyst = Agent(
    role="Currency Analyst",
    goal = "Provide real time currency analysis and financial insights.",
    backstory = (
        "You are a Finance Expert with deep knowledge of global exchange rates."
        "You help users with financial conversion and financial decision making"
    ),
    tools= [currency_tool], # Attach our custom tool
    verbose=True
)

In [ ]:
# Assign tast to the currency_analyst Agent
from crewai import Task
currency_conversion_task = Task(
    description=(
        "Convert {amount} {from_currency} to {to_currency}."
        "Using real time excahnge rates."
        "Provide the equivalent amount and "
        "explain any financial context."
    ),
    expected_output=("A detailed output imcluding the"
                     "converted amount and financial insights."),
    agent=currency_analyst
  )

In [ ]:
# Create a Crew, assign an Agent to the task and execute it.
from crewai import Crew, Process
crew = Crew(
    agents=[currency_analyst],
    tasks=[currency_conversion_task],
    process=Process.sequential,
    verbose=True
)
response=crew.kickoff(inputs= {"amount": 100,
                               "from_currency": "USD",
                               "to_currency": "EUR"})



╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3f9c8568-0288-45e8-b74a-0d23d5708611                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Currency Analyst                                                                                        │
│                                                                                                                 │
│  Task: Convert 100 USD to EUR.Using real time excahnge rates.Provide the equivalent amount and explain any      │
│  financial context.                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭─────────────────────────────────────────────────── LLM Error ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ LLM Call Failed                                                                                             │
│  Error: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please       │
│  check your plan and billing details. For more information on this error, read the docs:                        │
│  https://platform.openai.com/docs/guides/error-codes/api-errors.                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 47cfd0a9-60f9-4705-81aa-17510fb0ae59                                                                     │
│  Agent: Currency Analyst                                                                                        │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 3f9c8568-0288-45e8-b74a-0d23d5708611                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output:                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.

In [ ]:
# Create a Crew, assign an Agent to the task and execute it.
from crewai import Crew, Process
crew = Crew(
    agents=[currency_analyst],
    tasks=[currency_conversion_task],
    process=Process.sequential,
    verbose=True
)
response=crew.kickoff(inputs= {"amount": 100,
                               "from_currency": "USD",
                               "to_currency": "EUR"})

In [ ]:
# Create a Crew, assign an Agent to the task and execute it.
from crewai import Crew, Process
crew = Crew(
    agents=[currency_analyst],
    tasks=[currency_conversion_task],
    process=Process.sequential,
    verbose=True
)
response=crew.kickoff(inputs= {"amount": 100,
                               "from_currency": "USD",
                               "to_currency": "EUR"})

In [ ]:
# Create a Crew, assign an Agent to the task and execute it.
from crewai import Crew, Process
crew = Crew(
    agents=[currency_analyst],
    tasks=[currency_conversion_task],
    process=Process.sequential,
    verbose=True
)
response=crew.kickoff(inputs= {"amount": 100,
                               "from_currency": "USD",
                               "to_currency": "EUR"})

After adding your OpenAI API key to Colab's Secrets Manager with the name `OPENAI`, you can retrieve it and set the environment variable like this:

In [ ]:
from google.colab import userdata
import os

OPENAI_API_KEY = userdata.get('OPENAI')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# You can verify that the environment variable is set (optional)
# print(os.getenv("OPENAI_API_KEY"))